# A1 · Reducción raw (esorex)

**Spec:** [`docs/spec_A1_codex_raw_reduction.md`](../docs/spec_A1_codex_raw_reduction.md)  |  **Bloque:** A · Reducción  |  **Run de este set:** `ROXs12b_realigned`

Reduce los raw MUSE con esorex y combina las exposiciones en el cubo del objeto.

| | |
|---|---|
| **Entrada** | Raw MUSE + calibraciones |
| **Salida (QC/productos)** | cubo combinado (`cube_files` del config), `stages/stage00r_qc.json` |
| **Consume aguas abajo** | Todo el bloque B |


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
bash scripts/reduce_raw.sh
```


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage00r_qc.json', RUN_ID))


## Coste de ejecución (esorex)

> ⏱️ Dos números distintos, y conviene no confundirlos:
>
> - **Medido**: lo que costó *esta* ejecución, receta a receta, leído de los
>   `duration_s` de los manifiestos de noche, del `perexp_scipost_execution.json`
>   y del `elapsed_minutes` del combinado. Un paso reanudado desde un checkpoint
>   anterior no vuelve a cobrarse, así que el total medido puede quedar **por
>   debajo** del coste real de reducir desde cero. La celda lo dice cuando pasa.
> - **Desde cero**: la extrapolación con las constantes medidas sobre las cuatro
>   noches reales de los dos objetos.

En perfil `cascade` la calibración se paga **por noche** (un flat, un wavecal y un
lsf por cada noche observada), no una vez por objeto:

```
T(min) ≈ N_noches·(T_cal + T_std) + N_exp·(t_scibasic + t_scipost + t_combine)
```

con `T_cal ≈ 62 min` (34 si el LSF_PROFILE se reutiliza de archivo),
`T_std ≈ 6.2 min`, `t_scibasic ≈ 3.0`, `t_scipost ≈ 4.5` y `t_combine ≈ 1.24`
min/exposición. `muse_bias` no entra: se reutiliza el MASTER_BIAS de archivo.

**Nota:** scibasic/scipost paralelizan sobre los 24 IFUs (OpenMP), así que el
tiempo escala aprox. inverso al nº de núcleos; `cores_factor` ajusta respecto a
esta máquina base (=1.0). Las constantes viven en
`musepipe/reduction/a1_review.py`, no copiadas aquí.


In [ ]:
from musepipe.reduction import a1_review as a1

rt = a1.measured_runtime(RUN_ID)
tot = rt['totals']
print(f"perfil de reducción : {a1.reduction_profile(RUN_ID)}")
print(f"A1 corrió en el run : {a1.resolve_a1_run(RUN_ID)}")
print(f"noches              : {tot['n_nights']}")
print(f"exposiciones        : {tot['n_exposures']} reducidas, "
      f"{tot['n_combined']} en el cubo combinado")
print()
if not rt['nights']:
    print('[sin manifiestos de noche: el work-dir de la reducción no es legible '
          'desde aquí — declara `work_dir` en el config del run de A1]')
for night in rt['nights']:
    print(f"  noche {night['night']}  ({night['n_science']} exposiciones de ciencia)")
    for step in night['steps']:
        mins = '     —' if step['minutes'] is None else f"{step['minutes']:6.1f}"
        print(f"     {step['recipe']:<14s} {step['step']:<16s} {mins} min   "
              f"{step['status']}")
    print(f"     {'':<14s} {'subtotal':<16s} {night['minutes']:6.1f} min\n")
pe = rt['perexp']
print(f"  P2 scipost por exposición   {pe['minutes']:6.1f} min "
      f"({pe['n_exposures']} exp, mediana "
      f"{pe['median_minutes'] if pe['median_minutes'] is None else round(pe['median_minutes'], 2)}"
      f" min/exp, {pe['status']})")
for excluded in pe['excluded']:
    print(f"     excluida: {excluded}")
cm = rt['combine']
if cm['minutes']:
    print(f"  P3 combinado streaming      {cm['minutes']:6.1f} min ({cm['method']})")
print()
print(f"  TOTAL MEDIDO   {tot['measured_minutes']:7.1f} min "
      f"(~{tot['measured_minutes']/60:.1f} h)")
print(f"  DESDE CERO     {tot['from_scratch_minutes']:7.1f} min "
      f"(~{tot['from_scratch_minutes']/60:.1f} h)   "
      f"[{tot['n_nights']} noches, {tot['n_exposures']} exp, "
      f"lsf {'reutilizado' if tot['reuse_lsf'] else 'calculado'}]")
if rt['resumed_steps']:
    print(f"  ({rt['resumed_steps']} paso(s) reanudados desde checkpoint no se "
          'cobran en el medido)')
print()
print('Extrapolación a otros tamaños (mismas noches, lsf como en este objeto):')
for n in sorted({1, 5, 10, max(tot['n_exposures'], 1), 2 * max(tot['n_exposures'], 1)}):
    m = a1.estimate_esorex_runtime(n, max(tot['n_nights'], 1),
                                   reuse_lsf=tot['reuse_lsf'])
    print(f'  {n:3d} exp  ->  {m:6.0f} min  (~{m/60:.1f} h)')


## Mapa de la cadena de este objeto

Qué etapas están ejecutadas, en **qué run** vive el QC de cada una y con qué fecha. El reparto entre runs se declara en `chain` dentro de `runs/<run>/config/config.json` (clave `stage_runs`); una etapa marcada `no ejecutada` no es un error, es trabajo pendiente para este objeto. Un `!` (CROSS-OBJECT) sí es un problema: se estaría leyendo otro objeto.

Ver `docs/plan_multiobjeto_notebooks_2026-07-24.md`.


In [ ]:
nb.show_chain(RUN_ID)


## Ejecutar o auditar


In [ ]:
cmd, target_run, missing = nb.launch_command('A1', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
print('comando resuelto para este objeto:')
print('   ', cmd or '(sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a1_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzado en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage00r_qc.json', RUN_ID)
nb.show(qc, keys=['shape', 'sha', 'offset', 'esorex', 'muse'], title='A1')


## Verificaciones (V1–V6): qué comprueban y qué respondieron

El QC de A1 (`stage00r_qc.json`) no re-reduce: **verifica** que el cubo entregado es sano y trazable. Qué comprueba cada una:

| Check | Qué comprueba | Por qué importa |
|---|---|---|
| **V1** STAT | La extensión STAT (varianza) existe, es positiva y con pocos NaN, excluyendo los canales del láser AO y los spaxels de borde | Es la base de toda la propagación de error aguas abajo |
| **V2** estándar | Continuo del estándar frente a su curva de respuesta | Cierra la validación de flujo **relativa** (si falta, se cierra por otra vía en A4/M3 vs Gaia) |
| **V3** WCS | `CRVAL3` y paso espectral | Si deriva, los λ del cubo no son fiables |
| **V4** vs ADP | Correlación de la imagen de luz blanca del cubo propio con la del ADP de ESO, tras igualar PSF | Validación cruzada **independiente** de la reducción propia contra el producto oficial |
| **V5** espectro estelar | Razón del espectro de la primaria entre cubo y ADP | Comprueba la calibración en flujo a lo largo de todo el rango |
| **V6** máscara de cielo | La máscara de cielo de scipost no muerde las fuentes | Un cielo mal enmascarado se resta del objeto |

Los **resultados no se escriben aquí**: la celda siguiente los lee del QC de este objeto. Tres estados distintos y que no significan lo mismo:

- `ok` / `no` — hay veredicto booleano en el QC.
- `medido` — el QC publica el número (p.ej. `v4_adp_whitelight_corr`) pero no un veredicto; el número se imprime y el juicio es tuyo.
- `unavailable` / `no ejecutada` — **laguna de proveniencia**, no un fallo físico. En perfil `cascade` el `stage00r_qc.json` puede ser el esqueleto vacío que escribe el driver, con los booleanos a `false` porque nadie los rellenó: eso es *no medido*, no *falló*, y la celda lo distingue explícitamente.

Evidencia gráfica en la figura de más abajo.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('A1', 'stages/stage00r_qc.json'):
        from musepipe.reduction import a1_review as a1

        payload = a1.a1_qc_payload(RUN_ID)
        print(f"perfil de reducción : {a1.reduction_profile(RUN_ID)}")
        print(f"QC de A1            : run {a1.resolve_a1_run(RUN_ID)}"
              f"{'  ⚠ ESQUELETO VACÍO (verificaciones sin ejecutar)' if a1.is_qc_skeleton(payload) else ''}")
        print()
        for check in a1.verification_status(RUN_ID):
            print(f"{check['tag']} · {check['label']}")
            print(f"   -> {check['status']}")
            if check['message']:
                print(f"      {check['message']}")
            print()


## Esquema de la reducción realizada

Las fases que A1 ejecutó **para este objeto**, con sus entradas (arriba de cada caja) y sus salidas (abajo), al estilo de los flujos de esoreflex. No es un dibujo aparte que se pueda desincronizar: las entradas salen de `esorex_driver.DEFAULT_RECIPE_REQUIREMENTS` —la tabla que gobierna la construcción de los SOF— y las salidas, de las puertas de producto de la cascada. Duración y estado vienen de los manifiestos de cada noche.

Tres carriles: **P1** calibración y scibasic, *por noche*; **P2** scipost *por exposición*, con sus dos puertas humanas; **P3** el combinado. `muse_exp_combine` aparece en rojo porque forma parte de la historia: se planificó, corrió y no produjo cubo, y por eso existe el combinado propio.


In [ ]:
import matplotlib.pyplot as plt
from musepipe.reduction import a1_review as a1

phases = a1.a1_phase_graph(RUN_ID)
fig = a1.draw_phase_schematic(phases)
out = a1.save_figure(fig, RUN_ID, 'a1_reduction_schematic.png')
print(f'{len(phases)} fases  ->  {out}')
plt.show()


## Evidencia de las verificaciones

Arriba, las verificaciones que dejaron imagen: el mapa de NaN de **V1** y el **ajuste** de **V4** —el PSF-matching contra el ADP de ESO, con el cubo propio convolucionado al σ que maximiza la correlación, la referencia y la diferencia normalizada—. Abajo, las dos que están medidas *en cualquier objeto* porque las produce el combinado: el desplazamiento aplicado a cada exposición (con el círculo de repetibilidad del centroide) y la dispersión de `CRVAL3` entre exposiciones, en canales, que es la evidencia directa de **V3**.

Si un objeto no tiene V1–V6 corridas, los dos paneles de arriba lo dicen y los de abajo siguen informando: la figura nunca miente por omisión.


In [ ]:
import matplotlib.pyplot as plt
from musepipe.reduction import a1_review as a1

fig = a1.draw_verification_panel(RUN_ID)
out = a1.save_figure(fig, RUN_ID, 'a1_verifications.png')
print('->', out)
plt.show()


## Los cubos que entraron al combinado (N×5)

Una fila por cubo y cinco columnas: la **mediana** de todo el cubo, tres **bandas** (azul, centro y roja, esquivando el láser AO y las bandas telúricas fuertes) y el **canal de Hα**, situado con la RV sistémica declarada en el config —sin valor por defecto: si falta, la celda se niega—.

La escala está **centrada en el halo**: `asinh` con la anchura lineal fijada por el ruido medido en el anillo exterior, y los mismos límites para todas las filas de una columna. Eso es lo que convierte el mosaico en un diagnóstico —una exposición con peor seeing, con el halo desplazado o con menos flujo se ve comparando su fila con las demás—; autoescalar cada panel lo escondería.

El encuadre y el centro de la primaria no se recalculan aquí: se toman del plan del combinado, donde ya los midió `measure_primary_center` con MAOPPY. El mosaico enseña, literalmente, lo que entró a combinar.

Cuesta ~2 s por cubo la primera vez y queda cacheado en `runs/<run>/tables/a1_cube_panels.npz`.


In [ ]:
import matplotlib.pyplot as plt
from musepipe.reduction import a1_review as a1

# Los cubos de una reducción anterior del mismo objeto (`perexp_cubes`)
# están apagados a propósito: mezclarían dos reducciones en una figura.
USE_LEGACY_PEREXP = False

rows = a1.list_a1_cubes(RUN_ID, include_legacy=USE_LEGACY_PEREXP)
kinds = {r.kind for r in rows}
print(f'{len(rows)} cubo(s) en disco  ({", ".join(sorted(kinds)) or "ninguno"})')
if kinds == {'combined'}:
    print('   Solo queda el cubo combinado: los cubos por exposición de esta\n'
          '   reducción se purgaron tras combinar (ocupaban cientos de GB).\n'
          '   La fila única NO es un error, es lo que hay en disco.')
ha_A, rest_A, rv = a1.halpha_observed_A(RUN_ID)
print(f'Hα: reposo {rest_A:.2f} Å, RV {rv:+.1f} km/s -> observada {ha_A:.2f} Å')

panels = a1.build_panels(
    RUN_ID, rows,
    progress=lambda i, n, label: print(f'  [{i}/{n}] {label}', flush=True))
fig = a1.draw_cube_mosaic(panels, rows)
out = a1.save_figure(fig, RUN_ID, 'a1_cube_mosaic.png', dpi=110)
print('->', out)
plt.show()


## Decisiones y notas
- **Alineado por el centroide de la primaria medido exposición a exposición** (`stream_combine.measure_primary_center`, ajuste MAOPPY), no por `muse_exp_align`. Los dos objetos llegaron aquí por caminos distintos: en ROXs 12 B `exp_align` daba offsets espurios de hasta 3.305" (cross-match de speckles NFM), y en ROXs 42B b `muse_exp_combine` corrió sin llegar a producir cubo. La repetibilidad del centroide y los desplazamientos aplicados están en el QC del combinado y en la figura de verificaciones.
- El combinado es **propio** (`musepipe.reduction.stream_combine`): sigma-clipping por voxel en ventanas de canales, pesos por tiempo de exposición y recorte a 200 px. Se escribió porque `muse_exp_combine` no era viable con estos volúmenes.
- El semáforo de A1 **sale de las verificaciones de este objeto**, no de una nota fija: míralo en la celda de V1–V6 de arriba. `unavailable` y `no ejecutada` son lagunas de proveniencia documentadas, no fallos físicos.
- **Solo ROXs 12 B (histórico, 2026-07-19):** los 6 blockers duros del bloque A están cerrados → F1 realineado = `yellow`, 0 bloqueantes; agrupación BIAS aceptada (A2b) y telúrica justificada (A1b), con molecfit corroborando STD_TELLURIC (A1a). Este párrafo NO se ha rehecho para otros objetos.


## Checks


In [ ]:
print('open_issues A1 (no bloqueantes):')
for i, s in enumerate(qc.get('open_issues', []), 1):
    print(f'  {i}. {s}')


## Conclusión (registrada)

**A1 de ROXs 12 B — todo lo de abajo sale del QC de este objeto; `n/d` significa que esa etapa no lo dejó escrito, no que valga cero.**

- **Entorno:** esorex 3.13.10 / MUSE 2.10.16.
- **Datos:** modo NFM-AO-N, Prog 109.23B7.001;109.23B7.002 (OB 3445598;3444577). El número de noches y de exposiciones **no se copia aquí**: la celda de coste lo cuenta de los manifiestos de la reducción, que siguen siendo correctos aunque el `stage00r_qc.json` sea el esqueleto vacío del driver.
- **Cubo:** [3681, 200, 200] (z,y,x), fracción finita 0.9416, sha256 `9ff14963c342ff9c5d3dddbb617b413882419357cc71311ba9e5bc2ac44adad7`.
- **Estado del QC:** pass.
- **Coste:** el medido y el de una reducción desde cero están en la celda de coste, calculados sobre las noches y exposiciones reales de este objeto.

El cubo de A1 es la entrada de todo el bloque B. Nada de esto es paper-final sin declarar los caveats que las verificaciones dejen abiertos.
